In [2]:
import numpy as np
from scipy import optimize

### generating data

In [3]:
# function for multivariate hawkes process
# code taken from https://arxiv.org/pdf/2502.18979 Algorithm 1

# mu - d dimensional vector
# alpha - dxd dimensional matrix
# beta - scalar > 0
# time - scalar > 0
def simulation_by_cluster_representation(mu, alpha, beta, time):
    d = len(mu) # dimension

    # family tree is not part of the paper's algorithm
    family_trees = [{} for _ in range(d)] # timestamp of event: (parent variable, timestamp of parent). parents of immigrants are -1

    # initialization
    T = [[] for _ in range(d)] # list of all immigrants and descendants
    A = [[] for _ in range(d)] # temporary list of ancestors

    # immigrant simulation
    for j in range(d):
        k = np.random.poisson(lam=mu[j]*time) # number of immigrants of type j

        small_t = [[] for _ in range(d)] # t != T (from the paper's algorithm)
        for _ in range(k):
            small_t[j].append(np.random.uniform(low=0, high=time)) # small t is different from big T

        A[j] = small_t[j]
        T[j] = list(set(T[j] + A[j]))

        for immigrant in T[j]: # assigns the parent of each immigrant to -1
            family_trees[j][immigrant] = (j, -1)

    # helper function for while loop condition (self explanatory)
    def there_exists_at_least_one_j_st_Aj_neq_empty(A):
        for j in range(len(A)):
            if A[j] != []:
                return True
        return False

    # offspring simulation
    while there_exists_at_least_one_j_st_Aj_neq_empty(A):
        O = [[] for _ in range(d)] # offspring initialization
        for j in range(d):
            if A[j] != []:
                for l in range(len(A[j])):
                    for j_prime in range(d):
                        if alpha[j_prime][j] > 0:
                            k = np.random.poisson(lam=alpha[j_prime][j])
                            
                            # finds elapsed time for descendants
                            small_t = [[] for _ in range(d)]
                            for i in range(k):
                                small_t[j_prime].append(np.random.exponential(beta))

                            # adds elapsed time of descendant to the timestamp of its parent (A[j][l] is the parent)
                            a_plus_t = []
                            for i in range(k):
                                a_plus_t.append(A[j][l] + small_t[j_prime][i])
                            O[j_prime] = list(set(O[j_prime] + a_plus_t))

                            # assigns the parent of each descendant
                            for descendant in a_plus_t:
                                family_trees[j_prime][descendant] = (j, A[j][l])

                        T[j_prime] = list(set(T[j_prime] + O[j_prime])) # offsprings are added to T
        
        A = O # offspring become ancestors

    # remove events beyond T and sort
    for j in range(d):
        i = 0
        while i < len(T[j]): # removes events that occur after time limit
            if T[j][i] > time:
                T[j].pop(i)
                i -= 1
            i += 1
        T[j] = sorted(T[j])

        # remove events from family_tree if they're beyond T
        for key in list(family_trees[j].keys()):
            if key > time:
                family_trees[j].pop(key)
    
    return T, family_trees


In [4]:
# generate data

# model parameters
mu = [0.1, 0.1] # background intensity
alpha = [[0.3, 0.45],
         [0.35, 0.4]] # mutual excitation matrix; alpha_12 means that 1 is excited by 2
beta = 1 # decay; assume beta is the same for all variables
time = 500 # time

np.random.seed(0)
timestamps, family_trees = simulation_by_cluster_representation(mu, alpha, beta, time) # hawkes

num_params = len(alpha) # used for flattening/reshaping alpha matrix

for i in timestamps:
    print(len(i), i)

# for tree in family_trees:
#     print()
#     for key in sorted(tree.keys()):
#         print(key, ':', tree[key])

194 [9.394900218177572, 10.10919872016286, 30.112735814634917, 35.51802909894347, 43.56464985077035, 50.098052040814075, 52.44716258957319, 59.13721293446661, 59.66418455574389, 60.43168394231795, 64.46314882742665, 66.79293261029483, 66.97080937110205, 67.07262501411657, 67.8111501967233, 67.84890811870666, 69.3550527883918, 69.37784698905199, 71.6766437045232, 72.35039657009686, 73.17697633018865, 81.60608073090252, 105.19128053692044, 105.48423199167398, 122.48832394141951, 123.1862614924834, 124.96132185667291, 125.2031842340993, 125.51407629564955, 125.70888239897018, 132.27780605231348, 132.4272371439506, 134.17142293256398, 150.83640108798815, 157.71417546209193, 160.4871900037291, 160.88275559678252, 179.753950286893, 181.8553854713113, 191.72075941288884, 193.85025877761197, 207.3309699952618, 211.82739966945235, 212.96003380364013, 218.51597689967073, 218.79360563134625, 219.0800410506786, 219.30075673116016, 220.02928056562595, 220.3161896583122, 220.38104795002118, 220.4942

### optimizing

In [5]:
# function for log likelihood of multivariate hawkes process
# code taken from Laub's textbook "The Elements of Hawkes Processes" section 5.3 

# list of tuples (timestamp, variable that generated the timestamp)
def generate_timestamp_tuple(timestamps):
    timestamp_tuple = []
    for i in range(len(timestamps)):
        for time in timestamps[i]:
            timestamp_tuple.append((time, i))
    return sorted(timestamp_tuple, key=lambda x: x[0])

# from Laub's book
def calculate_intensity_matrix(alpha, timestamps, mu, beta, time):
    l = mu
    for t in timestamps:
        if t[0] < time:
            l = np.add(l, alpha[int(t[1].astype(np.int64))] * np.exp(-beta * (time - t[0]))) # alpha[t[1]] is a 1D matrix
    return l

# from Laub's book
def calculate_compensator_matrix(alpha, timestamps, mu, beta, time):
    l = mu * time
    for t in timestamps:
        if t[0] < time:
            l = np.add(l, (1/beta) * alpha[int(t[1].astype(np.int64))] * (1 - np.exp(-beta * (time - t[0]))))
    return l

# from Laub's book
def negative_log_likelihood(alpha, *args):
    timestamps, mu, beta, time, num_params = args
    timestamp_tuple = generate_timestamp_tuple(timestamps)
    alpha = alpha.reshape((num_params,num_params)) # alpha must be flattened and reshaped because optimize.fmin_l_bfgs_b requires a 1d array

    timestamp_tuple = np.array(timestamp_tuple)
    mu = np.array(mu)
    alpha = np.array(alpha)

    l = 0
    for t in timestamp_tuple:
        intensity = calculate_intensity_matrix(alpha, timestamp_tuple, mu, beta, t[0])
        l += np.log(intensity[int(t[1].astype(np.int64))])
    for k in range(len(mu)):
        compensator = calculate_compensator_matrix(alpha, timestamp_tuple, mu, beta, time)
        l -= compensator[k]
    return -l

In [6]:
# optimize multivariate hawkes parameters
# assume known mu and beta

alpha = np.array(alpha).flatten() # lbfgs requires flattening; will be unflattened later
args = (timestamps, mu, beta, time, num_params)

likelihood = negative_log_likelihood(alpha, *args) # gets likelihood
print(likelihood)

initial_guess = [0.5 for _ in alpha] # arbitrary initialization
bounds = []
for i in range(len(alpha)):
    bounds.append((0.0001, 1-0.0001)) # bounds is (0,1), exclusive
bounds = np.array(bounds)

# do the optimization, with auto gradient
print('\noptimize with scipy-calculated gradient:')
opt_auto_grad = optimize.fmin_l_bfgs_b(func=negative_log_likelihood,
                                       args=args,
                                       x0=initial_guess,
                                       bounds=bounds,
                                       approx_grad=True)
for res in opt_auto_grad:
    print(res)

print('\nactual alpha:')
print(alpha)

alpha = alpha.reshape((num_params,num_params)) # reshape alpha for future use

499.722290897133

optimize with scipy-calculated gradient:
[0.37219904 0.28270227 0.43399587 0.42409971]
493.65068584685423
{'grad': array([-0.000216  , -0.00052296,  0.00025011, -0.00019327]), 'task': 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH', 'funcalls': 60, 'nit': 10, 'warnflag': 0}

actual alpha:
[0.3  0.45 0.35 0.4 ]


### latent positions

In [7]:
# Generate latent variables according to Gaussian distribution with specified variance
# z_j ~ N(0,sigma_z^2 I_d)  Z: shape = (p, d)
def generate_latent_Z(p, d, sigma_z):
    Z = np.random.normal(0, sigma_z, size=(p, d))
    return Z

# Generate Theta matrix with entries distributed according to Gaussian distribution with specified
# variance centered at value determined by pairwise distance between latent positions
# theta_jk = N(alpha - |z_j-z_k|^2,sigma_theta^2)   Theta: shape = (p, p)
def generate_theta(Z, alpha, sigma_theta):
    p = Z.shape[0]
    Theta = np.zeros((p, p))
    for j in range(p):
        for k in range(p):
            Theta[j, k] = np.random.normal(alpha - np.sum((Z[j] - Z[k])**2), sigma_theta)
    return Theta

def logistic(theta):
    theta_tilde = np.copy(theta)
    for i in range(len(theta)):
        for k in range(len(theta[i])):
            theta_tilde[i][k] = 1 / (1 + np.exp(-theta[i][k])) # np.log(1 + np.exp(theta[i][k]))

    return theta_tilde

In [ ]:
np.random.seed(0)

p = 2 # number of latent nodes to generate
d = 2 # dimension of latent space
sigma_z = 1 # variance of latent space generation

alph = -2 # constant for theta generation; this is alpha but i don't want to confuse it with the hawkes alpha
sigma_theta = 1 # variance for theta generation

Z = generate_latent_Z(p, d, sigma_z)
print(Z)

theta = generate_theta(Z, alph, sigma_theta)
print(theta)

theta_tilde = logistic(theta)
print(theta_tilde)

t, _ = simulation_by_cluster_representation(mu, theta_tilde, beta, time) # hawkes process using theta_tilde
print(t)

[[1.76405235 0.40015721]
 [0.97873798 2.2408932 ]]
[[-0.13244201 -6.98230551]
 [-5.05493922 -2.15135721]]
[[0.46693781 0.0009273 ]
 [0.00633734 0.10420447]]
[[9.394900218177572, 9.652912980543634, 9.68840786876734, 10.10919872016286, 19.59389612716034, 30.112735814634917, 35.51802909894347, 43.56464985077035, 48.049203946981535, 48.13378343612312, 48.31347441191752, 48.4543868229639, 48.55063789653064, 48.57217160684617, 48.68411614399646, 48.8590806995324, 49.22997600734772, 51.022405374014035, 53.401891759496316, 55.18757058215257, 56.37510002815186, 58.44649147794253, 58.759715408252646, 59.13721293446661, 59.32550790936557, 59.58346576342982, 59.77203048715403, 59.802023730704086, 60.09828060658445, 60.178443348633785, 60.25801350109527, 60.44651130107628, 61.04172182706255, 64.46314882742665, 69.0914756743069, 71.6766437045232, 79.48479182275986, 80.65475894249813, 98.29118084002675, 104.43837804741734, 104.5254866314318, 105.19128053692044, 106.77887001039244, 122.21279600080138,

In [ ]:
def euclidean_norm(z1, z2):
    total = 0
    for i in range(len(z1)):
        total += (z1[i] - z2[i]) ** 2
    return total

# log(P(X | theta_tilde, theta))
def log_p_x_given_thetas(alpha, *args):
    return -np.log(1 + np.exp(-negative_log_likelihood(alpha, *args))) # uses the hawkes log likelihood

# log(P(theta | Z))
def log_p_theta_given_z(sigma_theta, theta, alph, z):
    p = 0
    for j in range(p):
        for k in range(j,p):
            p -= np.log(np.sqrt(2 * np.pi * sigma_theta))
            
            numerator = (theta[j][k] - (alph - euclidean_norm(z[j], z[k]))) ** 2
            denominator = 2 * sigma_theta
            p -= numerator / denominator
    return p

# log(P(Z))
def log_p_z(sigma_z, z):
    p = 0
    for j in range(p):
        for l in range(d):
            p -= np.log(np.sqrt(2 * np.pi * sigma_z))

            numerator = z[j][l] ** 2 # might be z[j][d-1]
            denominator = 2 * sigma_z
            p -= numerator / denominator
    return p

# log likelihood = log(P(X | theta_tilde, theta)) + log(P(theta | Z)) + log(P(Z))
def negative_complete_data_log_likelihood(z, *args):
    p, d, theta, alph, sigma_theta, sigma_z, alpha, timestamps, mu, beta, time, num_params = args
    z = z.reshape(p,d) # z requres reshaping

    args = (timestamps, mu, beta, time, num_params)
    return -(log_p_x_given_thetas(alpha, *args) + log_p_theta_given_z(sigma_theta, theta, alph, z) + log_p_z(sigma_z, z))

In [ ]:
alpha = np.array(alpha).flatten() # flatten alpha for optimize.fmin_l_bfgs_b
Z = np.array(Z).flatten() # flatten z for optimize.fmin_l_bfgs_b
args = (p, d, theta, alph, sigma_theta, sigma_z, alpha, t, mu, beta, time, int(len(alpha)/2)) # gathers variables
ll = negative_complete_data_log_likelihood(Z, *args)
print(ll)

initial_guess = [0.1 for _ in Z] # arbitrary initial guess

# do the optimization, with auto gradient
print('\noptimize with scipy-calculated gradient:')
opt_auto_grad = optimize.fmin_l_bfgs_b(func=negative_complete_data_log_likelihood,
                                       args=args,
                                       x0=initial_guess,
                                       approx_grad=True)
for res in opt_auto_grad:
    print(res)

print('\nactual Z:')
print(Z)

alpha = alpha.reshape((num_params,num_params)) # reshape alpha
Z = Z.reshape(p,d) # reshape Z

-0.0

optimize with scipy-calculated gradient:
[0.1 0.1 0.1 0.1]
-0.0
{'grad': array([0., 0., 0., 0.]), 'task': 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL', 'funcalls': 5, 'nit': 0, 'warnflag': 0}

actual Z:
[1.76405235 0.40015721 0.97873798 2.2408932 ]
